# Taller: Cálculo y validación de TF, DF, IDF y TF-IDF

In [ ]:
# Carga
import nltk
import re
from collections import Counter

nltk.download("gutenberg")
from nltk.corpus import gutenberg

def procesar_texto(texto):
    return re.findall(r"\b\w+\b", texto.lower())

# Cargar los 18 documentos
documentos = {
    doc_id: procesar_texto(gutenberg.raw(doc_id))
    for doc_id in gutenberg.fileids()
}

print("Documentos:", len(documentos))
print("Tokens totales:", sum(len(t) for t in documentos.values()))

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


Documentos: 18
Tokens totales: 2199219


In [ ]:
import pandas as pd

# Contar los términos por documento
frecuencias = {
    doc_id: Counter(tokens)
    for doc_id, tokens in documentos.items()
}

# Construir matriz TF
matriz_tf = pd.DataFrame.from_dict(
    frecuencias, orient="index"
).fillna(0).astype("int32")

matriz_tf = matriz_tf.reindex(
    sorted(matriz_tf.columns), axis=1
)

print("Dimensiones:", matriz_tf.shape)
display(matriz_tf.iloc[:5, :10])

Dimensiones: (18, 42099)


,0,00,000,00021053,00081429,00482129,01,02,1,10
austen-emma.txt,0,0,2,0,0,0,0,0,0,2
austen-persuasion.txt,0,0,0,0,0,0,0,0,3,1
austen-sense.txt,0,0,0,0,0,0,0,0,2,1
bible-kjv.txt,0,0,0,0,0,0,0,0,2783,2117
blake-poems.txt,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Método 1: DF directamente desde los documentos
df_directo = Counter()

for tokens in documentos.values():
    df_directo.update(set(tokens))

# Método 2: DF desde la matriz TF
matriz_binaria = (matriz_tf > 0).astype(int)

df_matriz = matriz_binaria.sum(axis=0)

# Comparar ambos resultados
comparacion = pd.DataFrame({
    "DF directo": pd.Series(df_directo),
    "DF matriz": df_matriz
})

comparacion["Coinciden"] = (
    comparacion["DF directo"] == comparacion["DF matriz"]
)

print("¿Todos coinciden?:", comparacion["Coinciden"].all())
print("Diferencias:", (~comparacion["Coinciden"]).sum())

display(comparacion.head(15))

¿Todos coinciden?: True
Diferencias: 0


,DF directo,DF matriz,Coinciden
0,1,1,True
00,1,1,True
000,3,3,True
00021053,1,1,True
00081429,1,1,True
00482129,1,1,True
01,1,1,True
02,1,1,True
1,9,9,True
10,7,7,True


In [ ]:
terminos = ["the", "emma", "whale", "love"]

print("MATRIZ TF")
display(matriz_tf[terminos].head(10))

print("MATRIZ BINARIA")
display(matriz_binaria[terminos].head(10))

print("COMPARACIÓN DF")
display(comparacion.loc[terminos])

MATRIZ TF


,the,emma,whale,love
austen-emma.txt,5201,865,0,117
austen-persuasion.txt,3329,1,0,42
austen-sense.txt,4105,0,0,77
bible-kjv.txt,64023,0,3,311
blake-poems.txt,439,0,0,29
bryant-stories.txt,3451,0,16,13
burgess-busterbrown.txt,660,0,0,1
carroll-alice.txt,1642,0,0,3
chesterton-ball.txt,4965,0,2,25
chesterton-brown.txt,4670,0,0,9


MATRIZ BINARIA


,the,emma,whale,love
austen-emma.txt,1,1,0,1
austen-persuasion.txt,1,1,0,1
austen-sense.txt,1,0,0,1
bible-kjv.txt,1,0,1,1
blake-poems.txt,1,0,0,1
bryant-stories.txt,1,0,1,1
burgess-busterbrown.txt,1,0,0,1
carroll-alice.txt,1,0,0,1
chesterton-ball.txt,1,0,1,1
chesterton-brown.txt,1,0,0,1


COMPARACIÓN DF


,DF directo,DF matriz,Coinciden
the,18,18,True
emma,2,2,True
whale,6,6,True
love,15,15,True


In [ ]:
import numpy as np

N = len(documentos)

def calcular_idf(df, N):
    return np.log(N / df) + 1

idf_manual = calcular_idf(df_matriz, N)

print("Documentos:", N)
display(idf_manual.head(10).round(4))

Documentos: 18


,0
0,3.8904
00,3.8904
000,2.7918
00021053,3.8904
00081429,3.8904
00482129,3.8904
01,3.8904
02,3.8904
1,1.6931
10,1.9445


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Utilizar los mismos documentos y el mismo orden
ids = matriz_tf.index.tolist()
corpus = [" ".join(documentos[doc_id]) for doc_id in ids]

vectorizador = TfidfVectorizer(
    smooth_idf=False,     # ln(N/DF) + 1
    norm=None,            # Sin normalización vectorial
    sublinear_tf=False,   # TF como conteo normal
    lowercase=False,      # Ya normalizamos anteriormente
    token_pattern=r"(?u)\b\w+\b"
)

matriz_sklearn = vectorizador.fit_transform(corpus)

# Obtener IDF de cada término
idf_sklearn = pd.Series(
    vectorizador.idf_,
    index=vectorizador.get_feature_names_out()
)

print("Dimensiones:", matriz_sklearn.shape)
display(idf_sklearn.head(10).round(4))

Dimensiones: (18, 42099)


,0
0,3.8904
00,3.8904
000,2.7918
00021053,3.8904
00081429,3.8904
00482129,3.8904
01,3.8904
02,3.8904
1,1.6931
10,1.9445


In [ ]:
comparacion_idf = pd.DataFrame({
    "DF": df_matriz,
    "IDF manual": idf_manual,
    "IDF sklearn": idf_sklearn
})

comparacion_idf["Diferencia"] = abs(
    comparacion_idf["IDF manual"] -
    comparacion_idf["IDF sklearn"]
)

coinciden = (
    comparacion_idf.notna().all().all()
    and np.allclose(
        comparacion_idf["IDF manual"],
        comparacion_idf["IDF sklearn"]
    )
)

print("¿Coinciden todos los IDF?:", coinciden)
print("Diferencia máxima:",
      comparacion_idf["Diferencia"].max())

display(
    comparacion_idf.loc[
        ["the", "emma", "whale", "love"]
    ].round(5)
)

¿Coinciden todos los IDF?: True
Diferencia máxima: 0.0


,DF,IDF manual,IDF sklearn,Diferencia
the,18,1.00000,1.00000,0.0
emma,2,3.19722,3.19722,0.0
whale,6,2.09861,2.09861,0.0
love,15,1.18232,1.18232,0.0


In [ ]:
terminos = ["the", "emma", "whale", "love"]

# TF-IDF manual
tfidf_manual = matriz_tf[terminos].mul(
    idf_manual[terminos], axis=1
)

# TF-IDF con Scikit-learn
columnas = [vectorizador.vocabulary_[t] for t in terminos]

tfidf_libreria = pd.DataFrame(
    matriz_sklearn[:, columnas].toarray(),
    index=ids,
    columns=terminos
)

print("¿Coinciden los pesos TF-IDF?:",
      np.allclose(tfidf_manual, tfidf_libreria))

display(tfidf_manual.head().round(4))

¿Coinciden los pesos TF-IDF?: True


,the,emma,whale,love
austen-emma.txt,5201.0,2765.5993,0.0000,138.3316
austen-persuasion.txt,3329.0,3.1972,0.0000,49.6575
austen-sense.txt,4105.0,0.0000,0.0000,91.0388
bible-kjv.txt,64023.0,0.0000,6.2958,367.7020
blake-poems.txt,439.0,0.0000,0.0000,34.2873
